In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import kagglehub

# Set default to white theme
pio.templates.default = "plotly_white"

In [ ]:
#Attempts to read the files from local storage
try:
    ratings = pd.read_csv('data/RAW_interactions.csv')
    recipes = pd.read_csv('data/RAW_recipes.csv')
#If the user does not have the files loaded into memory, downloads the latest version of the data from Kaggle
except:
    path = kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")
    
    ratings = pd.read_csv(path.replace('\\', '/') + '/RAW_interactions.csv')
    recipes = pd.read_csv(path.replace('\\', '/') + '/RAW_recipes.csv')

100%|███████████████████████████████████████████████████████████████████████████████| 267M/267M [00:12<00:00, 23.2MB/s]

Extracting files...


In [ ]:
ratings.head()

In [ ]:
#Sets the index of the recipe dataframe to the recipe ID
#This makes it easier to join this data with ratings metrics, like ratings counts per recipe and average rating
recipes = recipes.set_index('id')
recipes.head()

In [ ]:
#Counts the number of unique recipes, users, and ratings
n_recipes = len(recipes['id'].unique())
n_users = len(ratings['user_id'].unique())
n_ratings = len(ratings)

#Calculates the density of the ratings matrix
density = n_ratings / (n_recipes * n_users)
print(density)

In [ ]:
#Plots the distribution of ratings (out of 5 stars) vs the # of ratings with that score
fig = px.histogram(ratings, x = 'rating', 
                  title = '# of Recipes by Rating')
fig.show()

In [ ]:
#Calculates the # of unique ratings per user
ratings_per_user = ratings.groupby('user_id')['recipe_id'].count().to_frame('n_ratings')

#Plots an ECDF to show the distribution of ratings per user (Y % of users have posted at most X ratings)
fig = px.ecdf(ratings_per_user, x = 'n_ratings', log_x = True)

#Updates the ECDF to show exponents of 10 as x-axis labels
fig.update_layout(
    xaxis = dict(
        tickmode = 'array',
        tickvals = [0] + [10 ** i for i in range(5)],
        ticktext = ['0'] + ['10<sup>' + str(i) +'</sup>' for i in range(6)],
    ),
    xaxis_title = '# of Recipes Rated',
    yaxis_title = '% of Users'
)

fig.show()

In [ ]:
#Counts the number of ratings per recipe
recipe_info = ratings.groupby('recipe_id')['user_id'].count().to_frame('n_ratings')
#Joins the recipe counts with the recipes dataframe
recipe_info = recipe_info.join(recipes)

#Plots an ECDF to show the distribution of ratings per recipe (Y% of recipes have at most X ratings)
fig = px.ecdf(recipe_info, x = 'n_ratings', log_x = True)

#Updates the ECDF to show exponents of 10 as x-axis labels
fig.update_layout(
    xaxis = dict(
        tickmode = 'array',
        tickvals = [0] + [10 ** i for i in range(5)],
        ticktext = ['0'] + ['10<sup>' + str(i) +'</sup>' for i in range(6)],
    ),
    xaxis_title = '# of Ratings',
    yaxis_title = '% of Recipes'
)

fig.show()

In [ ]:
#Displays the 5 most popular recipes (by # of ratings)
display(recipe_info.nlargest(5, 'n_ratings'))

In [ ]:
import ast
#Converts each recipe's tags column into a list, expands each list to make each tag its own row, then counts the number of times each tag appears
tag_counts = recipe_info['tags'].apply(ast.literal_eval).explode().value_counts()
#Displays the top 5 most frequent tags
tag_counts = tag_counts.to_frame('n_recipes')
display(tag_counts.nlargest(5, 'n_recipes'))

In [ ]:
#Counts the number of times each ingredient is listed in the ingredients table
ingredient_counts = recipe_info['ingredients'].apply(ast.literal_eval).explode().value_counts()
#Displays the top 5 most frequent tags
ingredient_counts = ingredient_counts.to_frame('n_recipes')
display(ingredient_counts.nlargest(5, 'n_recipes'))

In [ ]:
def damped_mean(df, rating_col, grouping_col, damping_factor):
    #Calculates the average rating across all movies
    avg_rating = df[rating_col].mean()
    num = df.groupby(grouping_col)[rating_col].sum() + (avg_rating * damping_factor)
    denom = df.groupby(grouping_col)[rating_col].count() + damping_factor
    return num / denom

#Calculates the average rating of each recipe, with a damping factor of 10
damped_avg_rating = damped_mean(ratings, 'rating', 'recipe_id', 10).to_frame('damped_avg_rating')

In [ ]:
#Joins the damped average ratings with the recipe data, and displays the top 5 recipes by damped average rating
recipe_info = recipe_info.join(damped_avg_rating)
display(recipe_info.nlargest(5, 'damped_avg_rating'))

In [ ]:
from wordcloud import STOPWORDS
from wordcloud import WordCloud

#Creates a list out of all words in the recipe reviews
text = ' '.join(ratings.loc[ratings['rating'] == 1, 'review'].astype(str).tolist())

#Filters out any characters that aren't letters or spaces, then converts all to lowercase
text = re.sub(r'[^A-Za-z\s]', '', text).lower()

#Filters out all stopwords using the wordcloud library
stopwords = set(STOPWORDS)
text = ' '.join(word for word in text.split() if word not in stopwords)

#Plots the wordcloud
wordcloud = WordCloud(width = 800, height = 400, background_color = 'white', collocations=False).generate(text)

plt.figure(figsize = (10, 5))
plt.imshow(wordcloud, interpolation = 'bilinear')
plt.axis('off')
plt.title('Food.com Recipe Reviews Word Cloud')
plt.show()